# Train ASR — Viet_Bud500, Conformer + byte CTC

Notebook dùng `model/ASR/{config,model,train}.py`, **không dùng stage BRIDGE hoặc checkpoint SLU cũ**. Model được train từ đầu.

**License:** README của dataset mâu thuẫn giữa metadata CC-BY-NC-SA-4.0 và phần LICENSE Apache-2.0; cần xác minh với nguồn trước khi dùng thương mại.

In [1]:
RUN = {
    'prepared_dir': 'data/ASR/viet_bud500_prepared',
    'split_dir': 'splits/random42_test20',
    'split_random_state': 42,
    'run_dir': 'training_runs/ASR_BUD500_2X_RANDOM42_TEST20',
    'training_enabled': False,
    'evaluate_test': True,
    'resume_last': True,
    'max_epochs': 100,
    'patience': 6,
    'min_delta': 1e-4,
    'micro_batch': 16,
    'effective_batch': 48,
    'num_workers': 4,
    'seed': 42,
    'vram_probe_enabled': False,
    'probe_candidates': [1, 2, 4, 8, 16],
    'gpu_memory_fraction': 0.80,
}


## 1. Môi trường
Chọn kernel Python đang có CUDA PyTorch. Cần `torch`, `torchaudio` bản tương thích, `numpy`, `soundfile`, `pyarrow`, `psutil`, `tqdm`, `ipywidgets`. Không tự cài/thay môi trường trong notebook này.

Lúc tạo notebook máy có torch 2.13.0+cu130 / torchaudio 2.11.0+cpu: khác phiên bản; CPU smoke tests đã chạy nhưng nên cài cặp chính thức cùng release/backend vào môi trường riêng trước khi train. Notebook cảnh báo version mismatch; VRAM/preflight trên runtime thực tế vẫn bắt buộc.

In [2]:
import copy, gc, hashlib, json, math, os, random, sys, warnings
from pathlib import Path
import numpy as np
import psutil
import torch
import torchaudio
from torch.utils.data import DataLoader

def find_project_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'model/ASR/train.py').is_file() and (candidate / 'data/ASR').is_dir():
            return candidate
    raise FileNotFoundError('Cannot find ModelForValls root')

ROOT = find_project_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from model.ASR.config import create_config, validate_config, save_config
from model.ASR.model import build_model, load_for_inference, load_manifest_audio
from model.ASR.train import ASRDataset, ManifestRows, collate_asr, compute_ctc_loss, LengthBucketSampler, fit, validate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HARDWARE = {'cpu_logical': psutil.cpu_count(), 'cpu_physical': psutil.cpu_count(logical=False),
            'ram_gib': round(psutil.virtual_memory().total / 2**30, 2),
            'torch': torch.__version__, 'torchaudio': torchaudio.__version__, 'device': str(DEVICE)}
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(DEVICE)
    HARDWARE.update(gpu=props.name, vram_gib=props.total_memory / 2**30, bf16=torch.cuda.is_bf16_supported())
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    warnings.warn('CUDA not available: preflight works, but full training on CPU is not recommended')
if torch.__version__.split('+')[0].split('.')[:2] != torchaudio.__version__.split('+')[0].split('.')[:2]:
    warnings.warn('Torch/torchaudio releases differ; use a compatible official pair before training')
print(json.dumps(HARDWARE, indent=2))
random.seed(RUN['seed']); np.random.seed(RUN['seed']); torch.manual_seed(RUN['seed'])
torch.set_num_threads(min(4, psutil.cpu_count() or 1))
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(RUN['seed'])
PREPARED = ROOT / RUN['prepared_dir']
SPLIT_ROOT = PREPARED / RUN['split_dir']
RUN_ROOT = ROOT / RUN['run_dir']


C:\Users\ASUS\AppData\Local\Temp\ipykernel_23612\3215676368.py:35: UserWarning: Torch/torchaudio releases differ; use a compatible official pair before training
  warnings.warn('Torch/torchaudio releases differ; use a compatible official pair before training')


{
  "cpu_logical": 16,
  "cpu_physical": 10,
  "ram_gib": 31.63,
  "torch": "2.13.0+cu130",
  "torchaudio": "2.11.0+cpu",
  "device": "cuda",
  "gpu": "NVIDIA GeForce RTX 4060 Laptop GPU",
  "vram_gib": 7.99560546875,
  "bf16": true
}


## 2. Kiểm tra dữ liệu đã chuẩn bị
Pipeline chuẩn bị có thể chạy lại/resume theo shard, **không train**:
```bash
python -m model.ASR.train --prepare-bud500 data/ASR/viet_bud500 --output-dir data/ASR/viet_bud500_prepared --prepare-workers 4
```
Không chạy hai tiến trình chuẩn bị cùng output. `.preparation.lock` là khóa đang chạy; nếu máy bị tắt giữa chừng, chỉ xóa lock sau khi xác nhận PID cũ đã chết. Không xóa raw dataset. Cần khoảng 100 GB cho waveform mới cộng manifests.

`preparation_report.json` chỉ xuất hiện khi toàn bộ pipeline đã thành công; notebook từ chối output chưa hoàn tất. Hash manifest được kiểm tra toàn bộ; audio shard được kiểm tra kích thước. Có SHA256 trong từng `done.json` để kiểm tra bit-rot toàn shard khi cần (bước đó phải đọc ~100 GB).

Tái tạo split 20% trong **thư mục mới chưa tồn tại**, không xử lý lại waveform:
```bash
python -m model.ASR.train --resplit-bud500 data/ASR/viet_bud500_prepared --output-dir data/ASR/viet_bud500_prepared/splits/random42_test20 --test-fraction 0.20 --random-state 42
```
Lệnh từ chối ghi đè split đã tồn tại. NumPy RandomState/MT19937 hoán vị các nhóm SHA256 được sắp thứ tự ổn định; chọn nguyên nhóm đến đúng số mẫu test. 20% tính trên **toàn bộ 649.155 mẫu hợp lệ, bao gồm validation**, không phải 20% riêng train. Validation giữ nguyên thành viên; 3 mẫu không đủ frame vẫn cách ly ở bộ gốc. Manifest giữ `original_split` và `source_shard`; `split` là nhãn mới, không suy ra split từ sample ID hay tên waveform shard.

In [3]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if (PREPARED / '.preparation.lock').exists():
    raise RuntimeError('Dataset preparation still running or stale lock requires inspection')
REPORT = json.loads((PREPARED / 'preparation_report.json').read_text(encoding='utf-8'))
assert REPORT['status'] == 'complete' and REPORT['preparation_version'] == 1
SPLIT_REPORT = json.loads((SPLIT_ROOT / 'split_report.json').read_text(encoding='utf-8'))
assert SPLIT_REPORT['status'] == 'complete' and SPLIT_REPORT['split_version'] == 'bud500_group_random_ctc2_v1'
assert SPLIT_REPORT['random_state'] == RUN['split_random_state']
assert SPLIT_REPORT['source_preparation_report_sha256'] == sha256_file(PREPARED / 'preparation_report.json')
assert SPLIT_REPORT['requested_test_fraction'] == 0.20
assert SPLIT_REPORT['counts']['test'] == math.ceil(SPLIT_REPORT['total_eligible_samples'] * 0.20)
assert SPLIT_REPORT['counts']['validation'] == REPORT['counts']['validation']['eligible_ctc2']
assert sum(SPLIT_REPORT['counts'].values()) == SPLIT_REPORT['total_eligible_samples']
MANIFESTS = {split: SPLIT_ROOT / f'{split}.ctc2.jsonl' for split in ('train', 'validation', 'test')}
for split, path in MANIFESTS.items():
    assert sha256_file(path) == SPLIT_REPORT['manifests'][path.name]['sha256'], f'Changed manifest: {path}'
    assert sha256_file(PREPARED / path.name) == SPLIT_REPORT['source_manifest_sha256'][split]
    print(split, {'samples': SPLIT_REPORT['counts'][split],
                  'original_split_counts': SPLIT_REPORT['original_split_counts'][split]})
    assert SPLIT_REPORT['counts'][split] > 0
print('Test fraction of eligible corpus:', SPLIT_REPORT['actual_test_fraction'])
for shard in REPORT['shards']:
    prefix = '-'.join(shard['source']['name'].split('-')[:2])
    for name, size in shard['files'].items():
        assert (PREPARED / 'shards' / prefix / name).stat().st_size == size, f'Truncated shard: {prefix}/{name}'
print('Hours:', REPORT['total_hours'])
print('Speaker/session:', SPLIT_REPORT['speaker_session_disjointness'])
MAX_SAMPLES = max(shard['max_samples'] or 0 for shard in REPORT['shards'])
print('Max source duration:', MAX_SAMPLES / 16000, 'seconds')


train {'samples': 511824, 'original_split_counts': {'train': 505844, 'test': 5980}}
validation {'samples': 7500, 'original_split_counts': {'validation': 7500}}
test {'samples': 129831, 'original_split_counts': {'train': 128311, 'test': 1520}}
Test fraction of eligible corpus: 0.2
Hours: 462.07974786458334
Speaker/session: UNVERIFIED: no speaker/session metadata; only exact waveform groups are isolated
Max source duration: 16.031125 seconds


In [4]:
CONFIG = create_config()
CONFIG['speech_encoder'].update(subsampling_factor=2, gradient_checkpointing=True)
CONFIG['training'].update(
    batch_size=RUN['micro_batch'], valid_batch_size=RUN['micro_batch'],
    gradient_accumulation_steps=math.ceil(RUN['effective_batch'] / RUN['micro_batch']),
    max_epochs=RUN['max_epochs'], patience=RUN['patience'], min_delta=RUN['min_delta'],
    num_workers=min(RUN['num_workers'], 8, max(0, (psutil.cpu_count() or 2) - 2)),
    seed=RUN['seed'], learning_rate=1e-4, weight_decay=0.01,
    amp=('bf16' if torch.cuda.is_bf16_supported() else 'fp16') if DEVICE.type == 'cuda' else 'off',
    allow_unknown_speakers=True, metadata_audit=True, pin_memory=DEVICE.type == 'cuda',
    persistent_workers=True, prefetch_factor=2,
)
RESUME_PATH = RUN_ROOT / 'last.pt' if RUN['resume_last'] and (RUN_ROOT / 'last.pt').is_file() else None
if RESUME_PATH is not None:
    saved = torch.load(RESUME_PATH, map_location='cpu', weights_only=True)
    CONFIG = copy.deepcopy(saved['config'])  # Resume: no silent architecture/optimizer changes.
    assert CONFIG['speech_encoder']['subsampling_factor'] == 2
    assert saved['run_state']['manifest_hashes'] == {key: sha256_file(MANIFESTS[key]) for key in ('train', 'validation')}
    print('Resume epoch', saved['epoch'] + 1, 'from', RESUME_PATH)
    del saved
validate_config(CONFIG)
print(json.dumps(CONFIG['speech_encoder'], indent=2))
print(json.dumps(CONFIG['training'], indent=2))
print('Effective batch:', CONFIG['training']['batch_size'] * CONFIG['training']['gradient_accumulation_steps'])


Resume epoch 14 from D:\ModelForValls\training_runs\ASR_BUD500_2X_RANDOM42_TEST20\last.pt
{
  "d_model": 512,
  "layers": 8,
  "attention_heads": 8,
  "d_ff": 2048,
  "dropout": 0.1,
  "conv_kernel_size": 31,
  "subsampling_factor": 2,
  "subsampling_kernel": 3,
  "subsampling_padding": 1,
  "use_group_norm": true,
  "gradient_checkpointing": true
}
{
  "batch_size": 16,
  "gradient_accumulation_steps": 3,
  "learning_rate": 0.0001,
  "weight_decay": 0.01,
  "gradient_clip_norm": 1.0,
  "max_epochs": 100,
  "num_workers": 4,
  "amp": "bf16",
  "seed": 42,
  "scheduler": "cosine",
  "bucket_by_length": true,
  "allow_unknown_speakers": true,
  "metadata_audit": true,
  "valid_batch_size": 16,
  "pin_memory": true,
  "persistent_workers": true,
  "prefetch_factor": 2,
  "patience": 6,
  "min_delta": 0.0001
}
Effective batch: 48


## 3. VRAM probe tùy chọn — không cập nhật weights
Bật `vram_probe_enabled` để thử batch 1/2/4/8/16 trên **độ dài lớn nhất** của corpus, forward + CTC backward hai lần, cùng hai buffer float32 dự trữ dung lượng Adam moments. Không gọi `optimizer.step()`, không lưu checkpoint. Chỉ chọn batch nếu peak reserved memory dưới 80% VRAM. Đây là phép ước lượng, không bảo đảm tuyệt đối không OOM; nếu train vẫn OOM, giảm microbatch và tăng accumulation tương ứng. Tránh ứng dụng khác chiếm VRAM.

Không probe lại khi resume. Nếu thay batch/config, tạo run mới hoặc dùng migration có chủ đích; `fit()` kiểm tra config và hash manifest khi resume.

In [5]:
PROBE_RESULTS = []
if RUN['vram_probe_enabled'] and DEVICE.type == 'cuda' and RESUME_PATH is None:
    max_frames = (MAX_SAMPLES - CONFIG['audio']['n_fft']) // CONFIG['audio']['hop_length'] + 1
    selected = None
    for candidate in RUN['probe_candidates']:
        probe_model = mels = output = loss = moment_reserve = None
        try:
            gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
            probe_model = build_model(CONFIG).to(DEVICE).train()
            parameter_count = sum(p.numel() for p in probe_model.parameters())
            moment_reserve = torch.zeros(2 * parameter_count, device=DEVICE, dtype=torch.float32)
            mels = torch.randn(candidate, max_frames, CONFIG['audio']['feature_dim'], device=DEVICE)
            lengths = torch.full((candidate,), max_frames, device=DEVICE, dtype=torch.long)
            target_length = max(1, min(240, (max_frames + 1) // 2 - 1))
            target_ids = torch.arange(target_length, dtype=torch.long).remainder(200) + 1
            batch = {'targets': target_ids.repeat(candidate),
                     'target_lengths': torch.full((candidate,), target_length, dtype=torch.long),
                     'min_frames': torch.full((candidate,), target_length, dtype=torch.long)}
            for _ in range(2):
                probe_model.zero_grad(set_to_none=True)
                with torch.autocast('cuda', dtype=torch.bfloat16 if CONFIG['training']['amp'] == 'bf16' else torch.float16):
                    output = probe_model(mels, lengths)
                loss = compute_ctc_loss(output, batch)
                loss.backward()
                if not all(torch.isfinite(p.grad).all() for p in probe_model.parameters() if p.grad is not None):
                    raise FloatingPointError('Nonfinite probe gradients')
                torch.cuda.synchronize()
                output = loss = None
            peak = torch.cuda.max_memory_reserved()
            safe = peak < torch.cuda.get_device_properties(DEVICE).total_memory * RUN['gpu_memory_fraction']
            PROBE_RESULTS.append({'batch': candidate, 'peak_reserved_gib': peak / 2**30, 'safe': safe})
            if safe:
                selected = candidate
            else:
                break
        except torch.OutOfMemoryError:
            PROBE_RESULTS.append({'batch': candidate, 'safe': False, 'error': 'CUDA OOM'})
            break
        finally:
            probe_model = mels = output = loss = moment_reserve = None
            gc.collect(); torch.cuda.empty_cache()
    if selected is None:
        raise RuntimeError('No safe batch found; inspect CUDA usage/model before training')
    CONFIG['training'].update(batch_size=selected, valid_batch_size=selected,
        gradient_accumulation_steps=math.ceil(RUN['effective_batch'] / selected))
    validate_config(CONFIG)
    print(PROBE_RESULTS)
else:
    print('Probe skipped. Configured microbatch (measured previously on RTX 4060 8 GB):', CONFIG['training']['batch_size'])


Probe skipped. Configured microbatch (measured previously on RTX 4060 8 GB): 16


In [6]:
# Full metadata scan uses JSONL offsets rather than copying all dictionaries into Windows workers.
TRAIN_DATA = ASRDataset(MANIFESTS['train'], CONFIG)
VALID_DATA = ASRDataset(MANIFESTS['validation'], CONFIG)
for name, dataset in [('train', TRAIN_DATA), ('validation', VALID_DATA)]:
    audit = dataset.audit(metadata_only=True)
    assert audit['invalid_count'] == 0, audit['errors'][:10]
    print(name, 'eligible records', len(dataset), 'metadata audit passed')
    del audit

# Read actual packed audio, not only metadata. No backward/optimizer here.
example = TRAIN_DATA[0]
batch = collate_asr([example])
preflight_model = build_model(CONFIG).to(DEVICE).eval()
with torch.inference_mode():
    with torch.autocast(device_type=DEVICE.type, enabled=CONFIG['training']['amp'] != 'off',
                        dtype=torch.bfloat16 if CONFIG['training']['amp'] == 'bf16' else torch.float16):
        output = preflight_model(batch['mels'].to(DEVICE), batch['mel_lengths'].to(DEVICE))
    preflight_loss = compute_ctc_loss(output, batch).item()
print({'mels': tuple(batch['mels'].shape), 'logits': tuple(output['logits'].shape),
       'finite_initial_ctc_loss': preflight_loss, 'transcript': example['text']})
# Optional listening: display(Audio(waveform.numpy(), rate=16000)) using IPython.display.Audio.
row = TRAIN_DATA.rows[0]
waveform, rate = load_manifest_audio(row, MANIFESTS['train'].parent)
print('Waveform:', waveform.shape, waveform.dtype, rate)
del preflight_model, output, batch, waveform, TRAIN_DATA, VALID_DATA
gc.collect()
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()


C:\Users\ASUS\AppData\Local\Temp\ipykernel_23612\864460329.py:2: UserWarning: Speaker/session metadata unavailable: speaker-disjointness is NOT verified
  TRAIN_DATA = ASRDataset(MANIFESTS['train'], CONFIG)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_23612\864460329.py:3: UserWarning: Speaker/session metadata unavailable: speaker-disjointness is NOT verified
  VALID_DATA = ASRDataset(MANIFESTS['validation'], CONFIG)


train eligible records 511824 metadata audit passed
validation eligible records 7500 metadata audit passed
{'mels': (1, 142, 80), 'logits': (1, 71, 257), 'finite_initial_ctc_loss': 6.65189266204834, 'transcript': 'không được về nhà ăn tết thì là năm nay'}
Waveform: torch.Size([23042]) torch.float32 16000


## 4. Train và lưu checkpoint
`fit()` tạo AdamW + cosine scheduler, AMP, gradient accumulation/clipping, bucket theo độ dài, early stopping theo validation CER. CTC luôn float32, `zero_infinity=False`. Không dùng test để chọn checkpoint.

Lưu trong `training_runs/ASR_BUD500_2X_RANDOM42_TEST20/` (run mới cho split mới):
- `best.pt`: checkpoint có validation CER nhỏ nhất.
- `last.pt`: resume từ **cuối epoch hoàn tất gần nhất**, gồm optimizer/scheduler/scaler/RNG/config/hash manifests và early-stop state. Interrupt giữa epoch sẽ chạy lại epoch đó; không có mid-epoch resume.
- `metrics.jsonl`, `config.json`, `run.snapshot.json`.

**Không resume checkpoint từng train trên split cũ**: một phần dữ liệu train cũ giờ đã vào test mới, gây rò rỉ. Run mới được tách tên; hash manifests được kiểm tra khi resume. Trainer từ chối ghi đè run cũ khi không resume. `num_workers=4` mặc định; nếu Jupyter Windows gặp vấn đề spawn, dùng 0 hoặc chạy CLI trong terminal. Dataset/collate/worker-init đều nằm trong module importable, không khai báo trong cell.

Sau khi đổi `training_enabled=True`, chạy lại cell RUN và cell này (không cần probe lại). Chạy toàn bộ notebook mặc định **không bắt đầu train**.

In [7]:
if RUN['training_enabled']:
    if DEVICE.type != 'cuda':
        raise RuntimeError('Full training requires CUDA in this notebook; use CLI explicitly for CPU')
    RUN_ROOT.mkdir(parents=True, exist_ok=True)
    snapshot = {'run': RUN, 'hardware': HARDWARE, 'probe_results': PROBE_RESULTS,
                'prepared_report_sha256': sha256_file(PREPARED / 'preparation_report.json'),
                'split_report_sha256': sha256_file(SPLIT_ROOT / 'split_report.json'),
                'source_rows': REPORT['source_rows'], 'eligible_counts': SPLIT_REPORT['counts'],
                'split_random_state': SPLIT_REPORT['random_state']}
    (RUN_ROOT / 'run.snapshot.json').write_text(json.dumps(snapshot, ensure_ascii=False, indent=2), encoding='utf-8')
    save_config(CONFIG, RUN_ROOT / 'config.requested.json')
    trained_model = fit(CONFIG, MANIFESTS['train'], MANIFESTS['validation'], RUN_ROOT,
                        device=str(DEVICE), resume=RESUME_PATH)
    print('Best checkpoint:', RUN_ROOT / 'best.pt')
    print('Last checkpoint:', RUN_ROOT / 'last.pt')
    del trained_model
    gc.collect(); torch.cuda.empty_cache()
else:
    print('Preflight only. Set RUN["training_enabled"] = True to train explicitly.')


Preflight only. Set RUN["training_enabled"] = True to train explicitly.


## 5. Test cuối — bật riêng sau khi chọn model
Bật `evaluate_test=True` khi đã hoàn tất lựa chọn cấu hình/checkpoint. Metric dưới đây chấm trên **129.831 mẫu random holdout (20% corpus đủ frame CTC 2×)**, không phải official test 7.500 mẫu của Bud500 và không so trực tiếp như cùng benchmark. Ba mẫu CTC lỗi đã loại trước khi random split; báo denominator 649.155 và số loại trừ. Test tăng khoảng 17,3 lần nên đánh giá sẽ lâu hơn. Chưa kiểm chứng speaker-independent generalization; không dùng checkpoint train theo split cũ.

In [8]:
# Xem text dự đoán trên 15 audio test ngẫu nhiên; không tính loss/metrics toàn tập.
TEST_CASES = 15
TEST_SAMPLE_SEED = RUN.get('seed', 42)  # Đổi số này, hoặc đặt None, để lấy 15 mẫu khác.
TEST_TEXT_RESULTS = []
if RUN['evaluate_test']:
    checkpoint_path = RUN_ROOT / 'best.pt'
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f'Chưa có best checkpoint: {checkpoint_path}')
    checkpoint_metadata = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    try:
        if checkpoint_metadata.get('run_state', {}).get('manifest_hashes') != {
            key: sha256_file(MANIFESTS[key]) for key in ('train', 'validation')
        }:
            raise ValueError('Checkpoint dùng split khác; không test trên bộ chia này')
        best_epoch = checkpoint_metadata['epoch'] + 1  # Checkpoint lưu index từ 0.
    finally:
        del checkpoint_metadata
    test_manifest = MANIFESTS['test']
    if sha256_file(test_manifest) != SPLIT_REPORT['manifests'][test_manifest.name]['sha256']:
        raise ValueError('Test manifest đã thay đổi so với split report')
    test_rows = ManifestRows(test_manifest)
    if len(test_rows) != SPLIT_REPORT['counts']['test'] or not len(test_rows):
        raise ValueError('Số mẫu test không hợp lệ')
    selected_indices = random.Random(TEST_SAMPLE_SEED).sample(
        range(len(test_rows)), min(TEST_CASES, len(test_rows)))
    best_model = load_for_inference(checkpoint_path, DEVICE)
    print(f'Best epoch: {best_epoch} | {len(selected_indices)} case ngẫu nhiên | seed={TEST_SAMPLE_SEED}')
    print(f'Checkpoint: {checkpoint_path}')
    try:
        for case_number, index in enumerate(selected_indices, 1):
            row = test_rows[index]
            waveform, sample_rate = load_manifest_audio(
                row, test_manifest.parent, best_model.config['audio']['sample_rate'])
            # ASR nhận audio; transcript chỉ dùng đối chiếu, không đưa vào model.
            prediction = best_model.transcribe(waveform, sample_rate, request_id=row['sample_id'])
            TEST_TEXT_RESULTS.append({
                'case': case_number, 'sample_id': row['sample_id'],
                'reference_text': row['transcript'], 'predicted_text': prediction['text'],
                'status': prediction['status'],
            })
            print('\n' + '=' * 80)
            print(f'Case {case_number:02d}/{len(selected_indices)} | {row["sample_id"]}')
            print('Text gốc       :', row['transcript'])
            predicted_text = prediction['text']
            if predicted_text is None:
                predicted_text = f'<không có text hợp lệ: {prediction["status"]}>'
            elif not predicted_text.strip():
                predicted_text = '<text rỗng>'
            print('Model dự đoán  :', predicted_text)
            print('Trạng thái     :', prediction['status'])
            del waveform
    finally:
        del best_model, test_rows
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
else:
    print('Test text đang tắt; đặt RUN["evaluate_test"] = True rồi chạy lại cell này.')


Best epoch: 13 | 15 case ngẫu nhiên | seed=42
Checkpoint: D:\ModelForValls\training_runs\ASR_BUD500_2X_RANDOM42_TEST20\best.pt

Case 01/15 | viet_bud500:train:00068:002995
Text gốc       : ý của tôi vẫn như cũ tôi không quen sai
Model dự đoán  : ý cô tôi vẫn như cũ tôi không quen sai
Trạng thái     : ok

Case 02/15 | viet_bud500:train:00011:005477
Text gốc       : đàn ông bên cạnh mình nhưng anh đã nói
Model dự đoán  : đàn ông bên cạnh mình nhưng anh đã nói
Trạng thái     : ok

Case 03/15 | viet_bud500:train:00002:004076
Text gốc       : vân anh tròn mắt nhìn mạnh tuấn trời ơi
Model dự đoán  : vân anh tròn mắt nhìn mạnh tuấn trời ơi
Trạng thái     : ok

Case 04/15 | viet_bud500:train:00079:001710
Text gốc       : chứng nghĩa là cần phải có sức đợi chờ
Model dự đoán  : crứng nghĩa là cần phải có sức đợi chờ
Trạng thái     : ok

Case 05/15 | viet_bud500:train:00029:001843
Text gốc       : thì cái giới quản lý thị trường trung
Model dự đoán  : thì cái giới quản lý thị trường trung
Trạng t

## 6. Inference sau khi train
```python
from model.ASR.model import load_for_inference, load_audio
asr = load_for_inference(RUN_ROOT / 'best.pt', DEVICE)
waveform, sample_rate = load_audio('path/to/example.wav')
result = asr.transcribe(waveform, sample_rate)
print(result)
```
Không dùng output từ model khởi tạo ngẫu nhiên làm transcript. Lỗi UTF-8/input phải được xử lý riêng. Chưa có VAD/no-speech detector hay streaming.

### CLI tương đương
Sau khi đã lưu `config.requested.json`:
```bash
python -m model.ASR.train --config training_runs/ASR_BUD500_2X_RANDOM42_TEST20/config.requested.json --train-manifest data/ASR/viet_bud500_prepared/splits/random42_test20/train.ctc2.jsonl --validation-manifest data/ASR/viet_bud500_prepared/splits/random42_test20/validation.ctc2.jsonl --output-dir training_runs/ASR_BUD500_2X_RANDOM42_TEST20 --device cuda
```
Thêm `--resume training_runs/ASR_BUD500_2X_RANDOM42_TEST20/last.pt` để resume đúng run của split mới. Chạy terminal có thể thuận tiện hơn cho tác vụ nhiều giờ.